[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/AI_journey/blob/main/notebooks/12_gradient_boosting/04_tuning_and_rf_vs_gb.ipynb)

# Bài 04 — Tuning Gradient Boosting, Random Forest vs Gradient Boosting

**🎯 Mục tiêu**
- Quy trình tuning GB thực tế: cố định learning rate + early stopping → tune độ phức tạp cây → tune lấy mẫu & regularization → giảm learning rate.
- Dùng `RandomizedSearchCV` và (tuỳ chọn) **Optuna**.
- So sánh RF vs GB: chất lượng mặc định, độ nhạy với hyperparameter, nhiễu nhãn, chi phí.
- Failure case: overfit khi quá nhiều vòng, ngoại suy, ràng buộc đơn điệu (monotonic constraints); template model.

In [ ]:
# ⚙️ Setup — chạy được cả local lẫn Google Colab (local: đã `pip install -e .` thì bỏ qua)
import sys
if "google.colab" in sys.modules:
    !test -d /content/ml-course || git clone -q --depth 1 https://github.com/ThanhDatVN/AI_journey /content/ml-course
    !pip install -q -e /content/ml-course
    sys.path.insert(0, "/content/ml-course/src")
    %cd /content/ml-course/notebooks/12_gradient_boosting

In [ ]:
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import loguniform, randint, uniform
from sklearn.ensemble import HistGradientBoostingClassifier, HistGradientBoostingRegressor, RandomForestClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import RandomizedSearchCV, cross_val_score, train_test_split

from ml_course.datasets import make_customer_churn
from ml_course.visualization import set_style

set_style()
churn = make_customer_churn(n=8000, seed=33)
X = pd.get_dummies(churn.drop(columns=["customer_id", "churn"]), dtype=float)
y = churn["churn"].to_numpy()
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=0, stratify=y)
print(X_train.shape)

## 1. Overfit khi quá nhiều vòng — vì sao cần early stopping

In [ ]:
X_fit, X_val, y_fit, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=0, stratify=y_train)
model = HistGradientBoostingClassifier(learning_rate=0.3, max_iter=600, max_leaf_nodes=63, early_stopping=False,
                                       l2_regularization=0.0, random_state=0).fit(X_fit, y_fit)
staged = [roc_auc_score(y_val, p[:, 1]) for p in model.staged_predict_proba(X_val)]
staged_tr = [roc_auc_score(y_fit, p[:, 1]) for p in model.staged_predict_proba(X_fit)]
plt.plot(staged_tr, label="train")
plt.plot(staged, label="validation")
best = int(np.argmax(staged))
plt.axvline(best, color="gray", ls="--", label=f"tốt nhất: {best + 1} vòng")
plt.xlabel("số vòng (cây)")
plt.ylabel("ROC-AUC")
plt.legend()
plt.title("learning_rate lớn + cây lớn: overfit nhanh sau vài chục vòng")
plt.show()
print(f"validation tốt nhất = {max(staged):.4f} tại vòng {best + 1} | sau 600 vòng = {staged[-1]:.4f}")

## 2. Quy trình tuning gợi ý

1. Chọn `learning_rate` vừa phải (0.05–0.1), **early stopping** trên validation để chọn số cây.
2. Tune **độ phức tạp cây**: `max_depth` / `num_leaves` / `max_leaf_nodes`, `min_samples_leaf` (`min_child_samples`).
3. Tune **ngẫu nhiên hoá & regularization**: `subsample`, `colsample`, `l2_regularization` (`reg_lambda`).
4. Cuối cùng: **giảm** `learning_rate` (vd ÷ 2–5) và tăng số cây tương ứng → thường thêm một chút chất lượng.

In [ ]:
search = RandomizedSearchCV(
    HistGradientBoostingClassifier(learning_rate=0.1, max_iter=500, early_stopping=True, validation_fraction=0.15,
                                   n_iter_no_change=20, random_state=0),
    {"max_leaf_nodes": randint(4, 64), "min_samples_leaf": randint(5, 200),
     "l2_regularization": loguniform(1e-3, 10), "max_features": uniform(0.4, 0.6)},
    n_iter=20, cv=4, scoring="roc_auc", random_state=0)
start = time.perf_counter()
search.fit(X_train, y_train)
print(f"RandomizedSearch ({time.perf_counter() - start:.0f}s) tốt nhất:",
      {k: (round(v, 3) if isinstance(v, float) else v) for k, v in search.best_params_.items()},
      f"| ROC-AUC CV = {search.best_score_:.4f}")
final = HistGradientBoostingClassifier(learning_rate=0.03, max_iter=2000, early_stopping=True, validation_fraction=0.15,
                                       n_iter_no_change=50, random_state=0, **search.best_params_).fit(X_train, y_train)
print(f"giảm learning_rate → 0.03: dừng sau {final.n_iter_} vòng | ROC-AUC test = {roc_auc_score(y_test, final.predict_proba(X_test)[:, 1]):.4f}")

### (Tuỳ chọn) Optuna — Bayesian optimization

Optuna chọn cấu hình tiếp theo dựa trên kết quả các lần thử trước (TPE) và có thể **cắt sớm** lần thử kém (pruning). Chi tiết ở chương 16.

In [ ]:
try:
    import warnings

    warnings.filterwarnings("ignore", message="IProgress not found")
    import optuna

    optuna.logging.set_verbosity(optuna.logging.WARNING)

    def objective(trial):
        params = {"max_leaf_nodes": trial.suggest_int("max_leaf_nodes", 4, 64),
                  "min_samples_leaf": trial.suggest_int("min_samples_leaf", 5, 200, log=True),
                  "l2_regularization": trial.suggest_float("l2_regularization", 1e-3, 10, log=True),
                  "max_features": trial.suggest_float("max_features", 0.4, 1.0)}
        model = HistGradientBoostingClassifier(learning_rate=0.1, max_iter=500, early_stopping=True,
                                               validation_fraction=0.15, n_iter_no_change=20, random_state=0, **params)
        return cross_val_score(model, X_train, y_train, cv=4, scoring="roc_auc").mean()

    study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=0))
    study.optimize(objective, n_trials=20)
    print(f"Optuna tốt nhất: ROC-AUC CV = {study.best_value:.4f} |", {k: round(v, 3) if isinstance(v, float) else v for k, v in study.best_params.items()})
except ImportError:
    print("Chưa cài optuna — bỏ qua (pip install optuna).")

## 3. Random Forest vs Gradient Boosting

### (a) Mặc định vs đã tune

In [ ]:
rows = []
for name, model in [("RF mặc định", RandomForestClassifier(n_estimators=300, random_state=0)),
                    ("RF min_samples_leaf=10", RandomForestClassifier(n_estimators=300, min_samples_leaf=10, random_state=0)),
                    ("GB mặc định (HistGB)", HistGradientBoostingClassifier(random_state=0)),
                    ("GB đã tune", final)]:
    start = time.perf_counter()
    if model is not final:
        model.fit(X_train, y_train)
    rows.append({"model": name, "ROC-AUC test": roc_auc_score(y_test, model.predict_proba(X_test)[:, 1]),
                 "thời gian train (s)": time.perf_counter() - start if model is not final else np.nan})
pd.DataFrame(rows).round(4)

RF **mặc định** (`min_samples_leaf=1`) lại thấp nhất: trên nhãn nhiều nhiễu, lá chỉ 1 mẫu cho xác suất thô (gần 0/1) → xếp hạng kém. Chỉ một thay đổi đơn giản (`min_samples_leaf=10`) đã gần bằng GB đã tune. GB đã tune vẫn cao nhất.

### (b) Độ nhạy với hyperparameter

GB có nhiều "núm vặn" tương tác với nhau và **có thể hỏng nặng** nếu chọn sai (learning rate quá lớn, cây quá sâu, quá nhiều vòng). RF khó hỏng hơn nhiều. Phân phối ROC-AUC của 15 cấu hình **ngẫu nhiên** cho mỗi loại:

In [ ]:
rng = np.random.default_rng(1)
scores = {"Random Forest": [], "Gradient Boosting": []}
for _ in range(15):
    rf = RandomForestClassifier(n_estimators=150, max_features=rng.uniform(0.1, 1.0),
                                min_samples_leaf=int(rng.integers(1, 50)), random_state=0)
    gb = HistGradientBoostingClassifier(learning_rate=float(10 ** rng.uniform(-2.5, 0)), max_iter=int(rng.integers(20, 500)),
                                        max_leaf_nodes=int(rng.integers(2, 128)), min_samples_leaf=int(rng.integers(1, 50)),
                                        early_stopping=False, random_state=0)
    for name, m in [("Random Forest", rf), ("Gradient Boosting", gb)]:
        scores[name].append(roc_auc_score(y_test, m.fit(X_train, y_train).predict_proba(X_test)[:, 1]))
plt.boxplot(list(scores.values()), tick_labels=list(scores.keys()))
plt.ylabel("ROC-AUC test")
plt.title("15 cấu hình ngẫu nhiên: GB có đỉnh cao hơn nhưng trải rộng hơn")
plt.show()
for name, s in scores.items():
    print(f"{name:18s}: tốt nhất = {max(s):.4f} | tệ nhất = {min(s):.4f} | trung vị = {np.median(s):.4f}")

### (c) Tổng kết

| | Random Forest | Gradient Boosting |
|---|---|---|
| Cơ chế | cây sâu, độc lập, trung bình | cây nông, tuần tự, cộng dồn |
| Giảm | variance | bias (+ variance qua shrinkage/subsample) |
| Mặc định | tốt (tăng `min_samples_leaf` khi nhãn nhiễu), khó hỏng | tốt; cần tune để đạt đỉnh, dễ hỏng nếu chọn sai |
| Thêm cây | không overfit | có thể overfit → early stopping |
| Song song | hoàn toàn | trong từng cây (feature/histogram) |
| Nhiễu nhãn | robust | khá robust (log-loss), AdaBoost thì không |
| Thường thắng | baseline nhanh, ít thời gian tune | khi tune kỹ — "vua" của dữ liệu bảng trong các cuộc thi |

## 4. Failure case và các công cụ khắc phục

**(a) Ngoại suy** — như mọi model cây. **(b) Ràng buộc đơn điệu:** đôi khi ta *biết* quan hệ phải đơn điệu (giá nhà tăng theo diện tích); GB có thể học quan hệ "lởm chởm" ngược logic ở vùng ít dữ liệu. `monotonic_cst` ép buộc điều đó:

In [ ]:
houses = pd.read_csv("../../data/sample/house_prices.csv")
features = ["area_m2", "bedrooms", "bathrooms", "age_years", "distance_to_center_km"]
Xh, yh = houses[features].iloc[:300], houses["price_billion_vnd"].iloc[:300]      # ít dữ liệu → dễ học sai
probe = pd.DataFrame({"area_m2": np.linspace(30, 200, 200), "bedrooms": 3, "bathrooms": 2, "age_years": 10,
                      "distance_to_center_km": 8})
for cst, style in [(None, "r-"), ([1, 0, 0, -1, -1], "g-")]:
    m = HistGradientBoostingRegressor(max_iter=300, learning_rate=0.1, min_samples_leaf=5, monotonic_cst=cst, random_state=0).fit(Xh, yh)
    pred = m.predict(probe)
    n_drops = int(np.sum(np.diff(pred) < -1e-9))
    plt.plot(probe["area_m2"], pred, style, label=f"monotonic_cst = {cst} (số lần giảm: {n_drops})")
plt.xlabel("diện tích (m²)")
plt.ylabel("giá dự đoán (tỉ)")
plt.legend(fontsize=8)
plt.title("Ràng buộc: giá không được giảm khi diện tích tăng")
plt.show()

**(c) Leakage qua target encoding / feature tương lai** — GB khai thác mọi tín hiệu cực kỳ hiệu quả, kể cả tín hiệu rò rỉ → AUC "đẹp bất thường" là dấu hiệu cảnh báo (chương 05, bài 07).

## 5. Template model: Gradient Boosting

| Phần | Gradient Boosting (cây) |
|---|---|
| Problem | phân loại, hồi quy, ranking (LambdaMART), mọi loss khả vi |
| Intuition | cộng dồn nhiều cây nông, mỗi cây sửa lỗi còn lại của tổng trước |
| Geometry | tổng các hàm bậc thang → bậc thang mịn, bắt được tương tác bậc thấp (≈ độ sâu cây) |
| Probability | với log-loss: $F$ = log-odds, $p = \sigma(F)$ — calibration khá tốt nhưng nên kiểm tra |
| Objective | loss khả vi + regularization ($\lambda$, $\gamma$, shrinkage) |
| Optimization | gradient descent trong không gian hàm (XGBoost: Newton bậc hai) |
| Assumptions | ít; quan hệ xấp xỉ được bằng tổng các cây nông |
| Complexity | train $O(M \cdot n \cdot d)$ với histogram; dự đoán $O(M \cdot \text{độ sâu})$ |
| Hyperparameters | `learning_rate` × số cây (early stopping), độ sâu/số lá, `min_samples_leaf`, subsample/colsample, L2 |
| Preprocessing | không cần scaling; missing & categorical: hỗ trợ native (Hist GB, LightGBM, XGBoost, CatBoost) |
| Strengths | thường **chính xác nhất** trên dữ liệu bảng, loss linh hoạt, nhanh (histogram), xử lý missing |
| Weaknesses | nhiều hyperparameter, có thể overfit, tuần tự, không ngoại suy, nhạy leakage |
| Interpretation | SHAP (TreeSHAP), partial dependence, monotonic constraints |

## 🧠 Kiểm tra nhanh

1. Vì sao nên cố định learning rate và dùng early stopping trước khi tune các tham số khác?
2. Dữ liệu ít, cần model tốt trong 10 phút, không có thời gian tune: RF hay GB?
3. `monotonic_cst=[1, 0, -1]` nghĩa là gì?
4. Nêu 3 khác biệt giữa Random Forest và Gradient Boosting.

## 📌 Tóm tắt

- GB có thể overfit theo số vòng → early stopping là bắt buộc.
- Tuning: learning rate + early stopping → độ phức tạp cây → subsample/regularization → giảm learning rate.
- RF: mặc định tốt, khó hỏng; GB: đỉnh cao hơn khi tune, nhạy hyperparameter hơn.
- Monotonic constraints đưa hiểu biết miền vào model.

## 📚 Tài liệu tham khảo

- Friedman (2001) — *Greedy Function Approximation*
- Grinsztajn, Oyallon & Varoquaux (2022) — *Why do tree-based models still outperform deep learning on tabular data?*
- XGBoost / LightGBM docs — Parameters Tuning